# Stage 1: Data Ingestion & Master Label Taxonomy Mapping
## Explainable Hybrid Quantum-Classical Network Intrusion Detection System
---

**Scope:** Data loading, raw profiling, quality gating, and taxonomy label assignment ONLY.

**Stage 1 produces NO feature transformations.** It only assigns master labels, validates data quality, and emits structured records per network flow. All normalisation, encoding, scaling, SMOTE, and dimensionality reduction belong exclusively in Stage 2+.

**Master Labels:** NORMALL | DoSD | PROBE | EXPLOIT | MALWARE

**Target Datasets:** NSL-KDD (41 features), UNSW-NB15 (49 features), CICIDS-2017 (80+ features)

**Output Contract:** Per-record structured Parquet with 11 mandatory fields — raw features + metadata — ready for Stage 2.

**Reference:** Stage 1 Design Specification v1.0

---
## Step 1 — Environment Setup & Dependencies (Spec Step 1)

In [2]:
!pip install pandas numpy pyarrow -q

import pandas as pd
import numpy as np
import warnings
import json
import glob
import gc
from datetime import datetime, timezone

warnings.filterwarnings('ignore')

# ============================================================
# Pipeline Constants (Spec Step 1)
# ============================================================
SCHEMA_VERSION = "1.0"
QUANTUM_DIM = 8           # VQC qubit count — used in Stage 4; defined here for pipeline consistency
RANDOM_STATE = 42
QUARANTINE_THRESHOLD = 0.05  # 5% max quarantine rate per dataset

# Master label names per the Design Specification v1.0
MASTER_NAMES = {
    0: "NORMALL",
    1: "DoSD",
    2: "PROBE",
    3: "EXPLOIT",
    4: "MALWARE",
}

INGESTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()

print(f"Schema Version : {SCHEMA_VERSION}")
print(f"Quantum Dim    : {QUANTUM_DIM}")
print(f"Random State   : {RANDOM_STATE}")
print(f"Timestamp      : {INGESTION_TIMESTAMP}")
print(f"Master Labels  : {list(MASTER_NAMES.values())}")


Schema Version : 1.0
Quantum Dim    : 8
Random State   : 42
Timestamp      : 2026-03-11T02:32:41.807928+00:00
Master Labels  : ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---
## Step 2 — Dataset Acquisition & Loading (Spec Step 2)
Download and load all three datasets in their raw, untransformed state.

### 2.1 — NSL-KDD Dataset
CSV with no header row. 43 columns (41 features + label + difficulty_level).

In [5]:
# Download NSL-KDD dataset
import urllib.request

train_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt"
test_url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest+.txt"

urllib.request.urlretrieve(train_url, "KDDTrain+.txt")
urllib.request.urlretrieve(test_url, "KDDTest+.txt")

print("Download complete")
NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty_level",
]

nsl_train = pd.read_csv("KDDTrain+.txt", header=None, names=NSL_KDD_COLUMNS)
nsl_test = pd.read_csv("KDDTest+.txt", header=None, names=NSL_KDD_COLUMNS)
nsl_df = pd.concat([nsl_train, nsl_test], ignore_index=True)

print(f"NSL-KDD Train : {nsl_train.shape}")
print(f"NSL-KDD Test  : {nsl_test.shape}")
print(f"NSL-KDD Total : {nsl_df.shape}")
print(f"Columns       : {nsl_df.shape[1]} (expected 43)")
del nsl_train, nsl_test  # Free split DataFrames — only the combined nsl_df is needed
gc.collect()
nsl_df.head()


Download complete
NSL-KDD Train : (125973, 43)
NSL-KDD Test  : (22544, 43)
NSL-KDD Total : (148517, 43)
Columns       : 43 (expected 43)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


### 2.2 — UNSW-NB15 Dataset
Using Kaggle pre-split version (UNSW_NB15_training-set.csv + testing-set.csv). CSV with header row.

**Note (MN-03):** This uses the Kaggle pre-split version, not the original four CSV files (UNSW-NB15_1.csv through _4.csv). Class distributions should be verified against the original paper.

In [ ]:
!pip install kaggle -q
import os
import shutil

print("Configuring Kaggle API...")

# Create kaggle directory
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Copy kaggle.json
shutil.copy(r"ignore\kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))

# Set permission
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

print("Kaggle API configured successfully")

Configuring Kaggle API...
Kaggle API configured successfully



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Install Kaggle API (runs only if not installed)
import pandas as pd
import zipfile
import gc
import os

# Download dataset from Kaggle
!kaggle datasets download -d mrwellsdavid/unsw-nb15 -q

# Extract dataset
with zipfile.ZipFile("unsw-nb15.zip", 'r') as zip_ref:
    zip_ref.extractall("unsw_nb15_dataset")

# Load training and testing datasets
unsw_train = pd.read_csv("unsw_nb15_dataset/UNSW_NB15_training-set.csv")
unsw_test = pd.read_csv("unsw_nb15_dataset/UNSW_NB15_testing-set.csv")

# Combine datasets
unsw_df = pd.concat([unsw_train, unsw_test], ignore_index=True)

# Normalize attack_cat (handle missing values and remove whitespace)
if 'attack_cat' in unsw_df.columns:
    unsw_df['attack_cat'] = unsw_df['attack_cat'].fillna('').str.strip()

# Print dataset sizes
print(f"UNSW-NB15 Train : {unsw_train.shape}")
print(f"UNSW-NB15 Test  : {unsw_test.shape}")
print(f"UNSW-NB15 Total : {unsw_df.shape}")

# Free memory
del unsw_train, unsw_test
gc.collect()

# Preview dataset
unsw_df.head()

Dataset URL: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15
License(s): unknown
UNSW-NB15 Train : (82332, 45)
UNSW-NB15 Test  : (175341, 45)
UNSW-NB15 Total : (257673, 45)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label
0,1,0.000011,udp,-,INT,2,0,496,0,90909.0902,...,1,2,0,0,0,1,2,0,Normal,0
1,2,0.000008,udp,-,INT,2,0,1762,0,125000.0003,...,1,2,0,0,0,1,2,0,Normal,0
2,3,0.000005,udp,-,INT,2,0,1068,0,200000.0051,...,1,3,0,0,0,1,3,0,Normal,0
3,4,0.000006,udp,-,INT,2,0,900,0,166666.6608,...,1,3,0,0,0,2,3,0,Normal,0
4,5,0.000010,udp,-,INT,2,0,2126,0,100000.0025,...,1,3,0,0,0,2,3,0,Normal,0


### 2.3 — CICIDS-2017 Dataset (CF-04 Fix: Load ALL Daily Files)
Multiple daily CSV files must be concatenated. Column names have leading/trailing whitespace that must be stripped (D3-1).

In [12]:

import os
import glob
import zipfile

# Expected daily traffic files (Monday through Friday)
EXPECTED_DAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

# Download CICIDS-2017 dataset from Kaggle
!kaggle datasets download -d chethuhn/network-intrusion-dataset -q

# Extract dataset
with zipfile.ZipFile("network-intrusion-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("cicids2017_dataset")

print("Download complete.")

# Find extracted CSV files related to working hours
files = [f for f in glob.glob("cicids2017_dataset/**/*.csv", recursive=True)
         if "workinghours" in f.lower()]

print("Files extracted:", files)

Dataset URL: https://www.kaggle.com/datasets/chethuhn/network-intrusion-dataset
License(s): CC0-1.0
Download complete.
Files extracted: ['cicids2017_dataset\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'cicids2017_dataset\\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'cicids2017_dataset\\Friday-WorkingHours-Morning.pcap_ISCX.csv', 'cicids2017_dataset\\Monday-WorkingHours.pcap_ISCX.csv', 'cicids2017_dataset\\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'cicids2017_dataset\\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'cicids2017_dataset\\Tuesday-WorkingHours.pcap_ISCX.csv', 'cicids2017_dataset\\Wednesday-workingHours.pcap_ISCX.csv']


In [13]:
import glob

# Find ALL CSV files from the CICIDS-2017 download (CF-04: load ALL daily files)
# Use recursive glob to handle files in subdirectories after extraction
cicids_csv_files = sorted(glob.glob("**/*.csv", recursive=True) + glob.glob("*.csv"))

# Filter to only CICIDS-2017 daily files (case-insensitive — the dataset uses
# 'WorkingHours' on most days but 'workingHours' (lowercase w) on Wednesday)
cicids_csv_files = [
    f for f in cicids_csv_files
    if "workinghours" in f.lower()
]

# Deduplicate (in case recursive and non-recursive overlap)
cicids_csv_files = sorted(set(cicids_csv_files))

print(f"CICIDS-2017 CSV files found: {len(cicids_csv_files)}")
for f in cicids_csv_files:
    print(f"  {f}")

if not cicids_csv_files:
    raise FileNotFoundError("No CICIDS-2017 CSV files found after extraction.")

# Verify all expected days are present
for day in EXPECTED_DAYS:
    if not any(day.lower() in f.lower() for f in cicids_csv_files):
        print(f"  ⚠ WARNING: No file found for {day}!")
        if day == 'Wednesday':
            print(f"    → Wednesday contains DoSD (slowloris, slowhttptest, hulk, goldeneye) + Heartbleed")
            print(f"    → Missing Wednesday will significantly reduce DoSD class representation")

CICIDS-2017 CSV files found: 8
  cicids2017_dataset\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  cicids2017_dataset\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  cicids2017_dataset\Friday-WorkingHours-Morning.pcap_ISCX.csv
  cicids2017_dataset\Monday-WorkingHours.pcap_ISCX.csv
  cicids2017_dataset\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  cicids2017_dataset\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  cicids2017_dataset\Tuesday-WorkingHours.pcap_ISCX.csv
  cicids2017_dataset\Wednesday-workingHours.pcap_ISCX.csv


In [14]:
# Load and concatenate all CICIDS-2017 daily files
cicids_frames = []
for csv_file in cicids_csv_files:
    try:
        tmp = pd.read_csv(csv_file, low_memory=False, encoding='utf-8', encoding_errors='replace')
        # D3-1: Strip column name whitespace immediately
        tmp.columns = tmp.columns.str.strip()
        # Only accept files that have a 'Label' column (CICIDS-2017 schema marker)
        if 'Label' not in tmp.columns:
            # Some historical CICIDS versions use lowercase 'label'
            if 'label' in tmp.columns:
                tmp = tmp.rename(columns={'label': 'Label'})
                print(f"  Loaded {csv_file}: {tmp.shape} (renamed 'label' -> 'Label')")
            else:
                print(f"  SKIP  {csv_file}: {tmp.shape} — no 'Label' column, not a CICIDS-2017 file")
                continue
        else:
            print(f"  Loaded {csv_file}: {tmp.shape}, Labels: {tmp['Label'].nunique()}")
        cicids_frames.append(tmp)
    except Exception as e:
        print(f"  WARN: Could not load {csv_file}: {e}")

if not cicids_frames:
    raise ValueError("No CICIDS-2017 files with a 'Label' column were successfully loaded.")

# Verify column consistency across daily files (D3-2, VC-3.4)
col_sets = [set(f.columns) for f in cicids_frames]
common_cols = col_sets[0]
for cs in col_sets[1:]:
    common_cols = common_cols.intersection(cs)

all_cols = col_sets[0]
for cs in col_sets[1:]:
    all_cols = all_cols.union(cs)

if not common_cols or 'Label' not in common_cols:
    raise ValueError(
        f"Column intersection is empty or missing 'Label'. "
        f"Files may have incompatible schemas. "
        f"Per-file column counts: {[len(cs) for cs in col_sets]}"
    )

if common_cols != all_cols:
    diff = all_cols - common_cols
    print(f"\nWARNING: Column inconsistency across daily files.")
    print(f"  Common columns: {len(common_cols)}")
    print(f"  File-specific columns excluded: {diff}")
    # Use only the common column intersection (preserve order from first frame)
    ordered_common = [c for c in cicids_frames[0].columns if c in common_cols]
    cicids_frames = [f[ordered_common] for f in cicids_frames]

cicids_df = pd.concat(cicids_frames, ignore_index=True)
del cicids_frames  # Free the 5 individual daily DataFrames — only the combined DF is needed
gc.collect()

print(f"\nCICIDS-2017 Combined Shape: {cicids_df.shape}")
print(f"Columns: {list(cicids_df.columns[:10])}{'...' if cicids_df.shape[1] > 10 else ''}")
print(f"Attack types present: {cicids_df['Label'].nunique()}")


  Loaded cicids2017_dataset\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 79), Labels: 2
  Loaded cicids2017_dataset\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 79), Labels: 2
  Loaded cicids2017_dataset\Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 79), Labels: 2
  Loaded cicids2017_dataset\Monday-WorkingHours.pcap_ISCX.csv: (529918, 79), Labels: 1
  Loaded cicids2017_dataset\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 79), Labels: 2
  Loaded cicids2017_dataset\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 79), Labels: 4
  Loaded cicids2017_dataset\Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79), Labels: 3
  Loaded cicids2017_dataset\Wednesday-workingHours.pcap_ISCX.csv: (692703, 79), Labels: 6

CICIDS-2017 Combined Shape: (2830743, 79)
Columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet 

---
## Step 3 — Initial Dataset Audit (Spec Step 2)
Profile each dataset in its raw, untransformed state. Produces the Dataset Audit Report (VC-2.1).

In [16]:
def audit_dataset(df, name):
    """Produce a comprehensive audit report for a dataset."""
    print(f"\n{'='*60}")
    print(f"  DATASET AUDIT REPORT: {name}")
    print(f"{'='*60}")

    # Record count
    print(f"\n--- Record & Feature Summary ---")
    print(f"Total records  : {len(df):,}")
    print(f"Total features : {df.shape[1]}")

    # Data types
    dtypes = df.dtypes.value_counts()
    print(f"\n--- Data Types ---")
    for dtype, count in dtypes.items():
        print(f"  {dtype}: {count} columns")

    # Missing values
    missing = df.isnull().sum()
    missing_cols = missing[missing > 0]
    print(f"\n--- Missing Values ---")
    print(f"Columns with NaN: {len(missing_cols)}")
    if len(missing_cols) > 0:
        for col, cnt in missing_cols.items():
            print(f"  {col}: {cnt} ({cnt/len(df)*100:.2f}%)")

    # Infinity values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = {}
    for col in numeric_cols:
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            inf_counts[col] = inf_count
    print(f"\n--- Infinity Values ---")
    print(f"Columns with Inf: {len(inf_counts)}")
    for col, cnt in inf_counts.items():
        print(f"  {col}: {cnt} ({cnt/len(df)*100:.2f}%)")

    # Negative values in numeric columns (physical bounds check)
    neg_cols = []
    for col in numeric_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            neg_cols.append((col, neg_count))
    if neg_cols:
        print(f"\n--- Columns with Negative Values ---")
        for col, cnt in neg_cols[:10]:
            print(f"  {col}: {cnt}")
        if len(neg_cols) > 10:
            print(f"  ... and {len(neg_cols)-10} more columns")

    # Near-zero variance features
    print(f"\n--- Near-Zero Variance Features ---")
    nzv_found = False
    for col in numeric_cols:
        if df[col].nunique() <= 2 and df[col].var() < 0.01:
            print(f"  {col}: unique={df[col].nunique()}, var={df[col].var():.6f}")
            nzv_found = True
    if not nzv_found:
        print(f"  None detected")

    # Duplicate rows
    dup_count = df.duplicated().sum()
    print(f"\n--- Duplicates ---")
    print(f"Exact duplicate rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")

    # Numeric feature statistics
    print(f"\n--- Numeric Feature Statistics (summary) ---")
    if len(numeric_cols) > 0:
        print(df[numeric_cols].describe().round(2).to_string())
    else:
        print("  No numeric columns to describe.")

    return {
        'name': name,
        'records': len(df),
        'features': df.shape[1],
        'missing_cols': len(missing_cols),
        'inf_cols': len(inf_counts),
        'duplicates': dup_count,
    }

In [17]:
audit_nsl = audit_dataset(nsl_df, "NSL-KDD")


  DATASET AUDIT REPORT: NSL-KDD

--- Record & Feature Summary ---
Total records  : 148,517
Total features : 43

--- Data Types ---
  int64: 24 columns
  float64: 15 columns
  str: 4 columns

--- Missing Values ---
Columns with NaN: 0

--- Infinity Values ---
Columns with Inf: 0

--- Near-Zero Variance Features ---
  land: unique=2, var=0.000215
  root_shell: unique=2, var=0.001506
  num_outbound_cmds: unique=1, var=0.000000
  is_host_login: unique=2, var=0.000081

--- Duplicates ---
Exact duplicate rows: 610 (0.41%)

--- Numeric Feature Statistics (summary) ---
        duration     src_bytes     dst_bytes       land  wrong_fragment     urgent        hot  num_failed_logins  logged_in  num_compromised  root_shell  su_attempted   num_root  num_file_creations  num_shells  num_access_files  num_outbound_cmds  is_host_login  is_guest_login      count  srv_count  serror_rate  srv_serror_rate  rerror_rate  srv_rerror_rate  same_srv_rate  diff_srv_rate  srv_diff_host_rate  dst_host_count  dst_

In [18]:
audit_unsw = audit_dataset(unsw_df, "UNSW-NB15")


  DATASET AUDIT REPORT: UNSW-NB15

--- Record & Feature Summary ---
Total records  : 257,673
Total features : 45

--- Data Types ---
  int64: 30 columns
  float64: 11 columns
  str: 4 columns

--- Missing Values ---
Columns with NaN: 0

--- Infinity Values ---
Columns with Inf: 0

--- Near-Zero Variance Features ---
  None detected

--- Duplicates ---
Exact duplicate rows: 0 (0.00%)

--- Numeric Feature Statistics (summary) ---
              id        dur      spkts      dpkts       sbytes       dbytes        rate       sttl       dttl         sload        dload      sloss      dloss     sinpkt     dinpkt        sjit       djit       swin         stcpb         dtcpb       dwin     tcprtt     synack     ackdat      smean      dmean  trans_depth  response_body_len  ct_srv_src  ct_state_ttl  ct_dst_ltm  ct_src_dport_ltm  ct_dst_sport_ltm  ct_dst_src_ltm  is_ftp_login  ct_ftp_cmd  ct_flw_http_mthd  ct_src_ltm  ct_srv_dst  is_sm_ips_ports      label
count  257673.00  257673.00  257673.00  

In [19]:
audit_cicids = audit_dataset(cicids_df, "CICIDS-2017")


  DATASET AUDIT REPORT: CICIDS-2017

--- Record & Feature Summary ---
Total records  : 2,830,743
Total features : 79

--- Data Types ---
  int64: 54 columns
  float64: 24 columns
  str: 1 columns

--- Missing Values ---
Columns with NaN: 1
  Flow Bytes/s: 1358 (0.05%)

--- Infinity Values ---
Columns with Inf: 2
  Flow Bytes/s: 1509 (0.05%)
  Flow Packets/s: 2867 (0.10%)

--- Columns with Negative Values ---
  Flow Duration: 115
  Flow Bytes/s: 85
  Flow Packets/s: 115
  Flow IAT Mean: 115
  Flow IAT Max: 115
  Flow IAT Min: 2891
  Fwd IAT Min: 17
  Fwd Header Length: 35
  Bwd Header Length: 22
  Fwd Header Length.1: 35
  ... and 3 more columns

--- Near-Zero Variance Features ---
  Bwd PSH Flags: unique=1, var=0.000000
  Fwd URG Flags: unique=2, var=0.000111
  Bwd URG Flags: unique=1, var=0.000000
  RST Flag Count: unique=2, var=0.000242
  CWE Flag Count: unique=2, var=0.000111
  ECE Flag Count: unique=2, var=0.000243
  Fwd Avg Bytes/Bulk: unique=1, var=0.000000
  Fwd Avg Packets/Bul

### 3.1 — Native Label Vocabulary Extraction (VC-2.3)
Extract exact label string values from the data (not from documentation).

In [20]:
print("=" * 50)
print("  NATIVE LABEL VOCABULARY")
print("=" * 50)

print("\n--- NSL-KDD (column: label) ---")
nsl_labels = nsl_df['label'].value_counts()
print(nsl_labels.to_string())

print("\n--- UNSW-NB15 (column: attack_cat) ---")
unsw_labels = unsw_df['attack_cat'].value_counts()
print(unsw_labels.to_string())

print("\n--- CICIDS-2017 (column: Label) ---")
cicids_labels = cicids_df['Label'].value_counts()
print(cicids_labels.to_string())

# VC-2.4: Check MALWARE-mapped sample counts
print("\n--- MALWARE-Mapped Sample Counts (VC-2.4) ---")
unsw_malware = unsw_df['attack_cat'].str.strip().str.lower().isin(
    ['backdoors', 'worms', 'shellcode']
).sum()
cicids_malware = cicids_df['Label'].str.strip().isin(['Bot']).sum()
total_malware = unsw_malware + cicids_malware
print(f"  UNSW-NB15 (Backdoors+Worms+Shellcode): {unsw_malware}")
print(f"  CICIDS-2017 (Bot): {cicids_malware}")
print(f"  Total MALWARE-mapped samples: {total_malware}")
if total_malware < 500:
    print(f"  WARNING: MALWARE total ({total_malware}) < 500. Escalate for taxonomy review.")

  NATIVE LABEL VOCABULARY

--- NSL-KDD (column: label) ---
label
normal             77054
neptune            45871
satan               4368
ipsweep             3740
smurf               3311
portsweep           3088
nmap                1566
back                1315
guess_passwd        1284
mscan                996
warezmaster          964
teardrop             904
warezclient          890
apache2              737
processtable         685
snmpguess            331
saint                319
mailbomb             293
pod                  242
snmpgetattack        178
httptunnel           133
buffer_overflow       50
multihop              25
land                  25
rootkit               23
named                 17
ps                    15
sendmail              14
xterm                 13
imap                  12
ftp_write             11
loadmodule            11
xlock                  9
phf                    6
perl                   5
xsnoop                 4
spy                    2
worm      

---
## Step 4 — Schema Contract Validation (Spec Step 3)
Validate each dataset against its schema contract. D3-3: Raise on critical violations, warn on quality issues.

In [22]:
# ============================================================
# Schema Contracts per dataset
# MN-04 fix: expected_cols populated for all datasets
# ============================================================

SCHEMA_CONTRACTS = {
    'nsl_kdd': {
        'label_col': 'label',
        'expected_cols': 43,  # 41 features + label + difficulty_level
        # MF-04 + MF-NEW-02: near-zero variance features dropped
        # root_shell (var=0.001506) and is_host_login (var=0.000081) added per runtime audit
        'drop_cols': ['difficulty_level', 'land', 'urgent', 'num_outbound_cmds',
                      'root_shell', 'is_host_login'],
        'near_zero_variance': ['land', 'urgent', 'num_outbound_cmds', 'root_shell', 'is_host_login'],
        'categorical_cols': ['protocol_type', 'service', 'flag'],
        'identity_cols': [],
        'extra_label_drops': [],
    },
    'unsw_nb15': {
        'label_col': 'attack_cat',
        'expected_cols': 45,  # minimum expected after Kaggle pre-split format
        'drop_cols': ['id', 'srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime'],  # 'id' added for Kaggle version
        'near_zero_variance': [],
        'categorical_cols': ['proto', 'state', 'service'],
        'identity_cols': ['id', 'srcip', 'dstip', 'sport', 'dsport'],
        'extra_label_drops': ['label'],  # binary label column not needed for multi-class
    },
    'cicids2017': {
        'label_col': 'Label',
        'expected_cols': 78,  # minimum expected common columns
        'drop_cols': ['Flow ID', 'Source IP', 'Destination IP',
                      'Source Port', 'Destination Port', 'Timestamp'],
        'near_zero_variance': [],
        'categorical_cols': [],
        'identity_cols': ['Flow ID', 'Source IP', 'Destination IP',
                          'Source Port', 'Destination Port', 'Timestamp'],
        'extra_label_drops': [],
    },
}


def validate_schema(df, dataset_name, contract, strict_columns=False):
    """
    Validate a dataset against its schema contract.
    Raises on critical violations (D3-3).

    Args:
        strict_columns: If True, column count mismatch raises instead of warning.
    """
    violations = []
    critical = False

    # CRITICAL: Label column must exist
    if contract['label_col'] not in df.columns:
        violations.append(f"CRITICAL: Label column '{contract['label_col']}' not found")
        critical = True

    # Issue 5 fix: Column count check — escalated to REVIEW REQUIRED, optionally strict
    if contract['expected_cols'] and df.shape[1] < contract['expected_cols']:
        msg = (
            f"REVIEW REQUIRED: Expected >= {contract['expected_cols']} columns, "
            f"got {df.shape[1]}. Delta = {contract['expected_cols'] - df.shape[1]}. "
            f"Verify no essential features were lost during extraction."
        )
        violations.append(msg)
        if strict_columns:
            critical = True

    # MF-NEW-05: Columns to drop that are missing — document whether pre-stripped or renamed
    missing_drops = [c for c in contract['drop_cols'] if c not in df.columns]
    present_drops = [c for c in contract['drop_cols'] if c in df.columns]
    if missing_drops:
        violations.append(
            f"INFO: Columns to drop not found (assumed pre-stripped by Kaggle version): {missing_drops}"
        )
    if present_drops:
        violations.append(
            f"OK: Columns to drop that are present and will be removed: {present_drops}"
        )

    # INFO: Categorical columns present check
    missing_cat = [c for c in contract['categorical_cols'] if c not in df.columns]
    if missing_cat:
        violations.append(f"INFO: Categorical columns not found: {missing_cat}")

    print(f"\nSchema Validation — {dataset_name}")
    if violations:
        for v in violations:
            print(f"  {v}")
    else:
        print(f"  PASS — All schema checks passed")

    if critical:
        raise ValueError(f"Critical schema violation in {dataset_name}. Cannot proceed.")

    return violations


validate_schema(nsl_df, 'NSL-KDD', SCHEMA_CONTRACTS['nsl_kdd'])
validate_schema(unsw_df, 'UNSW-NB15', SCHEMA_CONTRACTS['unsw_nb15'])
validate_schema(cicids_df, 'CICIDS-2017', SCHEMA_CONTRACTS['cicids2017'])


Schema Validation — NSL-KDD
  OK: Columns to drop that are present and will be removed: ['difficulty_level', 'land', 'urgent', 'num_outbound_cmds', 'root_shell', 'is_host_login']

Schema Validation — UNSW-NB15
  INFO: Columns to drop not found (assumed pre-stripped by Kaggle version): ['srcip', 'dstip', 'sport', 'dsport', 'stime', 'ltime']
  OK: Columns to drop that are present and will be removed: ['id']

Schema Validation — CICIDS-2017
  INFO: Columns to drop not found (assumed pre-stripped by Kaggle version): ['Flow ID', 'Source IP', 'Destination IP', 'Source Port', 'Timestamp']
  OK: Columns to drop that are present and will be removed: ['Destination Port']


["INFO: Columns to drop not found (assumed pre-stripped by Kaggle version): ['Flow ID', 'Source IP', 'Destination IP', 'Source Port', 'Timestamp']",
 "OK: Columns to drop that are present and will be removed: ['Destination Port']"]

---
## Step 5 — Universal Data Quality Gate (Spec Step 4)
Apply dataset-agnostic quality validation. The ONLY transformations permitted are:
- QG-03: Replace Infinity with 99th percentile
- QG-04: Replace structural NaN with 0

All other quality issues are **flagged, not corrected**. Stage 2 owns corrections.

In [23]:
def apply_quality_gate(df, dataset_name, label_col, known_labels=None, inf_replacement_values=None):
    """
    Apply the Universal Quality Gate (QG-01 through QG-06).

    Args:
        df: Input DataFrame
        dataset_name: Name for logging
        label_col: Column name containing labels
        known_labels: Set of valid labels for QG-01
        inf_replacement_values: Optional dict {col: replacement_value} pre-computed
            on train-only data (from Stage 2). When provided, QG-03 uses these
            instead of computing p99 on the full dataset, eliminating leakage.

    Returns:
        df_clean: cleaned DataFrame with quality metadata columns preserved
        quarantine_df: quarantined records with reasons
        report: dict with quality gate statistics
    """
    # Issue 3 fix: reset to zero-based RangeIndex so positional indexing
    # into quality_flags_list is unconditionally safe regardless of upstream ops.
    df = df.reset_index(drop=True)

    n_original = len(df)
    quality_flags_list = [[] for _ in range(n_original)]
    has_missing = pd.Series(False, index=df.index)
    bounds_violation = pd.Series(False, index=df.index)
    quarantine_mask = pd.Series(False, index=df.index)
    quarantine_reasons = pd.Series('', index=df.index)

    print(f"\n{'='*60}")
    print(f"  QUALITY GATE: {dataset_name}")
    print(f"{'='*60}")
    print(f"Records in: {n_original:,}")

    # ---- QG-01: Label Validity ----
    if known_labels is not None:
        invalid_labels = ~df[label_col].isin(known_labels)
        n_invalid = invalid_labels.sum()
        if n_invalid > 0:
            quarantine_mask |= invalid_labels
            quarantine_reasons[invalid_labels] += 'QG-01;'
        print(f"QG-01 Label Validity   : {n_invalid} invalid labels quarantined")
    else:
        print(f"QG-01 Label Validity   : SKIPPED (no known label set provided)")

    # ---- QG-02: Feature Count ----
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    all_nan_rows = df[numeric_cols].isnull().all(axis=1)
    n_malformed = all_nan_rows.sum()
    if n_malformed > 0:
        quarantine_mask |= all_nan_rows
        quarantine_reasons[all_nan_rows] += 'QG-02;'
    print(f"QG-02 Feature Count    : {n_malformed} malformed rows quarantined")

    # ---- QG-03: Infinity Containment ----
    # VC-4.4 / MF-07: If inf_replacement_values is provided (from Stage 2 with
    # train-only computation), use those — no leakage. Otherwise compute p99 on the
    # data available (full dataset in Stage 1) and flag as leakage-prone.
    using_precomputed = inf_replacement_values is not None
    inf_replacements = {}
    for col in numeric_cols:
        inf_mask = np.isinf(df[col])
        if inf_mask.any():
            if using_precomputed and col in inf_replacement_values:
                replacement = inf_replacement_values[col]
            else:
                finite_vals = df.loc[~inf_mask, col]
                replacement = float(finite_vals.quantile(0.99))
            inf_replacements[col] = {'count': int(inf_mask.sum()), 'replacement': replacement}
            df.loc[inf_mask, col] = replacement
            for idx in df.index[inf_mask]:
                quality_flags_list[idx] = quality_flags_list[idx] + ['QG-03']
    n_inf_cols = len(inf_replacements)
    n_inf_total = sum(v['count'] for v in inf_replacements.values())
    print(f"QG-03 Infinity Handling : {n_inf_total} values in {n_inf_cols} columns replaced with {'precomputed' if using_precomputed else '99th pct'}")
    if inf_replacements:
        for col, info in inf_replacements.items():
            print(f"       {col}: {info['count']} -> {info['replacement']:.4f}")
        if not using_precomputed:
            print(f"       ⚠ WARNING: Replacement values computed on FULL dataset (VC-4.4).")
            print(f"         Stage 2 MUST recompute on train-only split to avoid data leakage.")
        else:
            print(f"       ✓ Using precomputed train-only replacement values (no leakage).")

    # ---- QG-04: NaN Classification ----
    # D4-2: structural NaN = NaN in statistical features (std, min, max, mean)
    nan_counts = df[numeric_cols].isnull().sum()
    nan_cols = nan_counts[nan_counts > 0]
    n_nan_structural = 0
    n_nan_missing = 0
    for col in nan_cols.index:
        nan_mask = df[col].isnull()
        col_lower = col.lower()
        is_structural = any(kw in col_lower for kw in ['std', 'min', 'max', 'mean', 'var'])
        if is_structural:
            df[col] = df[col].fillna(0)
            n_nan_structural += int(nan_mask.sum())
        else:
            has_missing.loc[nan_mask] = True
            n_nan_missing += int(nan_mask.sum())
            for idx in df.index[nan_mask]:
                quality_flags_list[idx] = quality_flags_list[idx] + ['QG-04']
    print(f"QG-04 NaN Classification: {n_nan_structural} structural (->0), {n_nan_missing} true missing (flagged)")

    # ---- QG-05: Physical Bounds ----
    bounds_keywords = ['bytes', 'count', 'duration', 'length', 'packets', 'rate']
    n_bounds = 0
    for col in numeric_cols:
        if any(kw in col.lower() for kw in bounds_keywords):
            neg_mask = df[col] < 0
            if neg_mask.any():
                bounds_violation.loc[neg_mask] = True
                n_bounds += int(neg_mask.sum())
                for idx in df.index[neg_mask]:
                    quality_flags_list[idx] = quality_flags_list[idx] + ['QG-05']
    print(f"QG-05 Physical Bounds  : {n_bounds} negative values flagged")

    # ---- QG-06: Duplicate Detection ----
    dup_mask = df.duplicated(keep='first')
    n_dups = int(dup_mask.sum())
    # Remove duplicates from all tracking structures
    keep_idx = ~dup_mask
    df = df[keep_idx].copy()
    has_missing = has_missing[keep_idx].copy()
    bounds_violation = bounds_violation[keep_idx].copy()
    quarantine_mask = quarantine_mask[keep_idx].copy()
    quarantine_reasons = quarantine_reasons[keep_idx].copy()
    quality_flags_list = [quality_flags_list[i] for i in range(len(keep_idx)) if keep_idx.iloc[i]]
    print(f"QG-06 Duplicate Detect : {n_dups} duplicates removed (one copy retained)")

    # Reset index again after duplicate removal so downstream positional indexing stays safe
    df = df.reset_index(drop=True)
    has_missing = has_missing.reset_index(drop=True)
    bounds_violation = bounds_violation.reset_index(drop=True)
    quarantine_mask = quarantine_mask.reset_index(drop=True)
    quarantine_reasons = quarantine_reasons.reset_index(drop=True)

    # ---- Apply quarantine ----
    quarantine_df = df[quarantine_mask].copy()
    if len(quarantine_df) > 0:
        quarantine_df['quarantine_reason'] = quarantine_reasons[quarantine_mask].values
    df_clean = df[~quarantine_mask].copy()

    # MF-05 FIX: Preserve quality metadata columns in output — do NOT drop them
    clean_mask = ~quarantine_mask
    clean_flags = [quality_flags_list[i] for i in range(len(clean_mask)) if clean_mask.iloc[i]]
    df_clean['quality_pass'] = True
    df_clean['quality_flags'] = [json.dumps(f) for f in clean_flags]
    df_clean['has_missing'] = has_missing[clean_mask].values
    df_clean['bounds_violation'] = bounds_violation[clean_mask].values

    quarantine_rate = len(quarantine_df) / n_original if n_original > 0 else 0
    print(f"\n--- Quality Gate Summary ---")
    print(f"Records in       : {n_original:,}")
    print(f"Duplicates removed: {n_dups:,}")
    print(f"Quarantined      : {len(quarantine_df):,} ({quarantine_rate*100:.2f}%)")
    print(f"Records out      : {len(df_clean):,}")

    if quarantine_rate > QUARANTINE_THRESHOLD:
        print(f"ALERT: Quarantine rate {quarantine_rate*100:.2f}% exceeds threshold {QUARANTINE_THRESHOLD*100}%!")

    report = {
        'dataset': dataset_name,
        'records_in': n_original,
        'duplicates_removed': n_dups,
        'quarantined': len(quarantine_df),
        'quarantine_rate': quarantine_rate,
        'records_out': len(df_clean),
        'inf_replacements': inf_replacements,
        'used_precomputed_inf': using_precomputed,
    }

    return df_clean, quarantine_df, report

In [24]:
# Apply quality gate to NSL-KDD
nsl_known_labels = set(nsl_df['label'].unique())
nsl_clean, nsl_quarantine, nsl_qg_report = apply_quality_gate(
    nsl_df.copy(), 'NSL-KDD', 'label', nsl_known_labels
)
del nsl_df  # Raw DF consumed by quality gate — nsl_clean carries the validated output



  QUALITY GATE: NSL-KDD
Records in: 148,517
QG-01 Label Validity   : 0 invalid labels quarantined
QG-02 Feature Count    : 0 malformed rows quarantined
QG-03 Infinity Handling : 0 values in 0 columns replaced with 99th pct
QG-04 NaN Classification: 0 structural (->0), 0 true missing (flagged)
QG-05 Physical Bounds  : 0 negative values flagged
QG-06 Duplicate Detect : 610 duplicates removed (one copy retained)

--- Quality Gate Summary ---
Records in       : 148,517
Duplicates removed: 610
Quarantined      : 0 (0.00%)
Records out      : 147,907


In [25]:
# Apply quality gate to UNSW-NB15
unsw_known_labels = set(unsw_df['attack_cat'].dropna().str.strip().unique())
unsw_clean, unsw_quarantine, unsw_qg_report = apply_quality_gate(
    unsw_df.copy(), 'UNSW-NB15', 'attack_cat', unsw_known_labels
)
del unsw_df  # Raw DF consumed by quality gate — unsw_clean carries the validated output



  QUALITY GATE: UNSW-NB15
Records in: 257,673
QG-01 Label Validity   : 0 invalid labels quarantined
QG-02 Feature Count    : 0 malformed rows quarantined
QG-03 Infinity Handling : 0 values in 0 columns replaced with 99th pct
QG-04 NaN Classification: 0 structural (->0), 0 true missing (flagged)
QG-05 Physical Bounds  : 0 negative values flagged
QG-06 Duplicate Detect : 0 duplicates removed (one copy retained)

--- Quality Gate Summary ---
Records in       : 257,673
Duplicates removed: 0
Quarantined      : 0 (0.00%)
Records out      : 257,673


In [26]:
# Apply quality gate to CICIDS-2017
cicids_known_labels = set(cicids_df['Label'].dropna().str.strip().unique())
cicids_clean, cicids_quarantine, cicids_qg_report = apply_quality_gate(
    cicids_df.copy(), 'CICIDS-2017', 'Label', cicids_known_labels
)
del cicids_df  # Raw DF consumed by quality gate — cicids_clean carries the validated output
gc.collect()   # Force GC here — CICIDS raw DF is the largest object in memory



  QUALITY GATE: CICIDS-2017
Records in: 2,830,743
QG-01 Label Validity   : 0 invalid labels quarantined
QG-02 Feature Count    : 0 malformed rows quarantined
QG-03 Infinity Handling : 4376 values in 2 columns replaced with 99th pct
       Flow Bytes/s: 1509 -> 12333333.3333
       Flow Packets/s: 2867 -> 2000000.0000
       ⚠ WARNING: Replacement values computed on FULL dataset (VC-4.4).
         Stage 2 MUST recompute on train-only split to avoid data leakage.
QG-04 NaN Classification: 0 structural (->0), 1358 true missing (flagged)
QG-05 Physical Bounds  : 2443148 negative values flagged
QG-06 Duplicate Detect : 308381 duplicates removed (one copy retained)

--- Quality Gate Summary ---
Records in       : 2,830,743
Duplicates removed: 308,381
Quarantined      : 0 (0.00%)
Records out      : 2,522,362


0

### 5.1 — Save Quarantine Files (VC-4.2)

In [27]:
# Persist quarantine records to Parquet with quarantine_reason field
if len(nsl_quarantine) > 0:
    nsl_quarantine.to_parquet('nsl_kdd_quarantine.parquet', index=False)
    print(f"Saved nsl_kdd_quarantine.parquet ({len(nsl_quarantine)} records)")
else:
    print("NSL-KDD: No quarantined records")

if len(unsw_quarantine) > 0:
    unsw_quarantine.to_parquet('unsw_nb15_quarantine.parquet', index=False)
    print(f"Saved unsw_nb15_quarantine.parquet ({len(unsw_quarantine)} records)")
else:
    print("UNSW-NB15: No quarantined records")

if len(cicids_quarantine) > 0:
    cicids_quarantine.to_parquet('cicids2017_quarantine.parquet', index=False)
    print(f"Saved cicids2017_quarantine.parquet ({len(cicids_quarantine)} records)")
else:
    print("CICIDS-2017: No quarantined records")

NSL-KDD: No quarantined records
UNSW-NB15: No quarantined records
CICIDS-2017: No quarantined records


---
## Step 6 — Drop Excluded Columns & Separate Features/Labels
Remove identity/leakage columns and near-zero variance features per schema contracts.

**MF-04 fix:** NSL-KDD now drops `land`, `urgent`, `num_outbound_cmds` (near-zero variance).

**MF-05 fix:** Quality metadata columns (`quality_pass`, `quality_flags`, `has_missing`, `bounds_violation`) are **preserved** in the output, not dropped.

In [28]:
def prepare_dataset(df_clean, contract):
    """
    Drop excluded columns and separate features from labels.
    Quality metadata columns are preserved.
    Returns: feature DataFrame (X), label Series (y), original_label Series
    """
    df = df_clean.copy()

    # Drop schema-excluded columns (identity, leakage, near-zero variance)
    cols_to_drop = [c for c in contract['drop_cols'] if c in df.columns]
    df = df.drop(columns=cols_to_drop, errors='ignore')
    print(f"Dropped schema-excluded columns: {cols_to_drop}")

    # Separate label column
    label_col = contract['label_col']
    original_label = df[label_col].copy()

    # Drop label and extra label columns from features
    label_drops = [label_col] + contract.get('extra_label_drops', [])
    # Also separate quality metadata columns (they go into output record, not features)
    quality_meta_cols = ['quality_pass', 'quality_flags', 'has_missing', 'bounds_violation']
    quality_meta = df[[c for c in quality_meta_cols if c in df.columns]].copy()

    X = df.drop(columns=[c for c in label_drops + quality_meta_cols if c in df.columns], errors='ignore')

    print(f"Features (X) shape : {X.shape}")
    print(f"Unique labels      : {original_label.nunique()}")

    return X, original_label, quality_meta


print("\n--- NSL-KDD ---")
nsl_X, nsl_original_label, nsl_quality_meta = prepare_dataset(
    nsl_clean, SCHEMA_CONTRACTS['nsl_kdd']
)

print("\n--- UNSW-NB15 ---")
unsw_X, unsw_original_label, unsw_quality_meta = prepare_dataset(
    unsw_clean, SCHEMA_CONTRACTS['unsw_nb15']
)

print("\n--- CICIDS-2017 ---")
cicids_X, cicids_original_label, cicids_quality_meta = prepare_dataset(
    cicids_clean, SCHEMA_CONTRACTS['cicids2017']
)

# Free memory — cleaned DataFrames fully consumed into (X, label, quality_meta) triplets
del nsl_clean, unsw_clean, cicids_clean
gc.collect()



--- NSL-KDD ---
Dropped schema-excluded columns: ['difficulty_level', 'land', 'urgent', 'num_outbound_cmds', 'root_shell', 'is_host_login']
Features (X) shape : (147907, 36)
Unique labels      : 40

--- UNSW-NB15 ---
Dropped schema-excluded columns: ['id']
Features (X) shape : (257673, 42)
Unique labels      : 10

--- CICIDS-2017 ---
Dropped schema-excluded columns: ['Destination Port']
Features (X) shape : (2522362, 77)
Unique labels      : 15


0

---
## Step 7 — Master Label Taxonomy Mapping (Spec Step 5)
Map all native labels to the unified 5-class master label system.

| Master Label | Meaning |
|---|---|
| NORMALL | Legitimate traffic |
| DoSD | Denial of Service / DDoS |
| PROBE | Reconnaissance / Port Scan / Fuzzing |
| EXPLOIT | R2L, U2R, Brute Force, Web, Buffer Overflow |
| MALWARE | Backdoors, Worms, Botnet, Shellcode |

**D5-6:** Generic (UNSW-NB15) **excluded** — no agreed real-world threat analogue. This replaces the earlier D5-4 provisional mapping to EXPLOIT (Low confidence). ~58,871 records removed; UNSW-NB15 output drops to ~198,802 records.

**MF-02 fix:** Unknown labels are quarantined, not silently assigned a default class.

**MN-02 fix:** Infiltration (CICIDS-2017) excluded per spec recommendation D5-3 (~36 samples).


In [29]:
def map_taxonomy_nsl_kdd(labels):
    """
    NSL-KDD taxonomy mapping. Returns master_label, mapping_confidence, flag_for_review.
    Classes present after mapping: NORMALL, DoSD, PROBE, EXPLOIT (no MALWARE in NSL-KDD).
    """
    DOS_LABELS = {
        "back", "land", "neptune", "pod", "smurf", "teardrop",
        "apache2", "udpstorm", "processtable", "mailbomb",
    }
    PROBE_LABELS = {"ipsweep", "nmap", "portsweep", "satan", "mscan", "saint"}
    EXPLOIT_LABELS = {
        # R2L
        "ftp_write", "guess_passwd", "imap", "multihop", "phf", "spy",
        "warezclient", "warezmaster", "sendmail", "named",
        "snmpgetattack", "snmpguess", "xlock", "xsnoop", "worm",
        # U2R
        "buffer_overflow", "loadmodule", "perl", "rootkit",
        "httptunnel", "ps", "sqlattack", "xterm",
    }

    master_labels = []
    confidences = []
    flags = []

    for label in labels:
        l = str(label).lower().strip().rstrip(".")
        if l == "normal":
            master_labels.append("NORMALL")
            confidences.append("High")
            flags.append(False)
        elif l in DOS_LABELS:
            master_labels.append("DoSD")
            confidences.append("High")
            flags.append(False)
        elif l in PROBE_LABELS:
            master_labels.append("PROBE")
            confidences.append("High")
            flags.append(False)
        elif l in EXPLOIT_LABELS:
            master_labels.append("EXPLOIT")
            confidences.append("High")
            flags.append(False)
        else:
            # MF-02: Unknown labels — do NOT silently assign a default class
            master_labels.append(None)  # Will be quarantined
            confidences.append(None)
            flags.append(True)

    return master_labels, confidences, flags


def map_taxonomy_unsw_nb15(labels):
    """
    UNSW-NB15 taxonomy mapping. Uses attack_cat column.
    D5-6: Generic -> EXCLUDED — no agreed real-world threat analogue.
    """
    UNSW_MAP = {
        "normal":         ("NORMALL", "High",   False),
        "":               ("NORMALL", "High",   False),  # empty attack_cat = normal
        "dos":            ("DoSD",    "High",   False),
        "reconnaissance": ("PROBE",   "High",   False),
        "fuzzers":        ("PROBE",   "Medium", False),  # D5-1: reconnaissance-first behaviour
        "analysis":       ("PROBE",   "Medium", False),  # D5-2: port/service enumeration
        "exploits":       ("EXPLOIT", "High",   False),
        "generic":        ("__EXCLUDE__", None, True),   # D5-6: excluded — no agreed real-world threat analogue
        "backdoors":      ("MALWARE", "High",   False),
        "backdoor":       ("MALWARE", "High",   False),  # handle singular variant
        "shellcode":      ("MALWARE", "High",   False),
        "worms":          ("MALWARE", "High",   False),
    }

    master_labels = []
    confidences = []
    flags = []

    for label in labels:
        l = str(label).lower().strip()
        if l in UNSW_MAP:
            ml, conf, flag = UNSW_MAP[l]
            master_labels.append(ml)
            confidences.append(conf)
            flags.append(flag)
        else:
            # MF-02: Unknown labels quarantined
            master_labels.append(None)
            confidences.append(None)
            flags.append(True)

    return master_labels, confidences, flags


import re

def _normalise_cicids_label(label):
    """
    Normalise CICIDS-2017 label strings to handle encoding variants.
    CF-NEW-01 fix: The 'Web Attack' labels contain separator characters that vary
    depending on how pandas decoded the CSV:
      - \\x96 (Windows-1252 byte) in the original CSV
      - \\u2013 (Unicode en-dash) when decoded as Latin-1/CP-1252
      - \\ufffd (Unicode replacement char) when decoded as UTF-8 with errors='replace'
      - '-' (plain hyphen) in some file versions
    This function normalises ALL separator variants to a plain hyphen before lookup.
    """
    # Replace en-dash (U+2013), em-dash (U+2014), replacement char (U+FFFD),
    # non-breaking hyphen (U+2011), figure dash (U+2012), and CP-1252 \\x96
    return re.sub(r'[\u2013\u2014\ufffd\u2011\u2012\x96]', '-', str(label).strip())


def map_taxonomy_cicids2017(labels):
    """
    CICIDS-2017 taxonomy mapping.
    D5-3: Infiltration EXCLUDED (~36 samples, statistically unreliable).
    D5-5: Heartbleed -> EXPLOIT (CVE, not malware).
    CF-NEW-01: Labels are normalised before lookup to handle encoding variants.
    """
    # All keys use plain hyphen '-' — the normaliser converts any variant before lookup
    CICIDS_MAP = {
        "BENIGN":                        ("NORMALL", "High",   False),
        "DoS slowloris":                 ("DoSD",    "High",   False),
        "DoS Slowhttptest":              ("DoSD",    "High",   False),
        "DoS Hulk":                      ("DoSD",    "High",   False),
        "DoS GoldenEye":                 ("DoSD",    "High",   False),
        "DDoS":                          ("DoSD",    "High",   False),
        "PortScan":                      ("PROBE",   "High",   False),
        "FTP-Patator":                   ("EXPLOIT", "High",   False),
        "SSH-Patator":                   ("EXPLOIT", "High",   False),
        "Web Attack - Brute Force":      ("EXPLOIT", "High",   False),
        "Web Attack - XSS":              ("EXPLOIT", "High",   False),
        "Web Attack - Sql Injection":    ("EXPLOIT", "High",   False),
        "Heartbleed":                    ("EXPLOIT", "High",   False),
        "Bot":                           ("MALWARE", "High",   False),
    }

    # D5-3: Infiltration is explicitly EXCLUDED
    EXCLUDED_LABELS = {"Infiltration"}

    master_labels = []
    confidences = []
    flags = []

    for label in labels:
        normalised = _normalise_cicids_label(label)
        if normalised in EXCLUDED_LABELS:
            master_labels.append("__EXCLUDE__")
            confidences.append(None)
            flags.append(False)
        elif normalised in CICIDS_MAP:
            ml, conf, flag = CICIDS_MAP[normalised]
            master_labels.append(ml)
            confidences.append(conf)
            flags.append(flag)
        else:
            master_labels.append(None)
            confidences.append(None)
            flags.append(True)

    return master_labels, confidences, flags


In [30]:
# ============================================================
# Apply taxonomy mapping to all three datasets
# ============================================================

def apply_taxonomy_and_build_output(X, original_label, quality_meta, dataset_name, map_fn):
    """
    Apply taxonomy mapping and build the Stage 1 output record with all 11 mandatory fields.
    Unknown/excluded labels are removed and counted.
    """
    # Runtime alignment assertion: X, original_label, and quality_meta must have
    # the same row count coming out of prepare_dataset(). If they don't, something
    # upstream changed rows independently and the positional join is unsafe.
    assert len(X) == len(original_label) == len(quality_meta), (
        f"Row count mismatch: X={len(X)}, labels={len(original_label)}, "
        f"quality_meta={len(quality_meta)}. Cannot safely align."
    )

    master_labels, confidences, flags = map_fn(original_label.values)

    # Build the output DataFrame: raw features + metadata
    output = X.copy()
    output = output.reset_index(drop=True)

    # 11 mandatory output schema fields
    output['source_dataset'] = dataset_name
    output['original_label'] = original_label.values
    output['master_label'] = master_labels
    output['mapping_confidence'] = confidences
    output['flag_for_review'] = flags

    # Quality metadata from quality gate (MF-05: preserved, not dropped)
    quality_meta = quality_meta.reset_index(drop=True)
    for col in ['quality_pass', 'quality_flags', 'has_missing', 'bounds_violation']:
        if col in quality_meta.columns:
            output[col] = quality_meta[col].values
        else:
            if col == 'quality_pass':
                output[col] = True
            elif col == 'quality_flags':
                output[col] = '[]'
            else:
                output[col] = False

    output['ingestion_timestamp'] = INGESTION_TIMESTAMP
    output['schema_version'] = SCHEMA_VERSION

    # Count and remove unmapped labels (MF-02: quarantine, don't default)
    unmapped_mask = output['master_label'].isnull()
    n_unmapped = unmapped_mask.sum()
    if n_unmapped > 0:
        unmapped_labels = output.loc[unmapped_mask, 'original_label'].value_counts()
        print(f"  QUARANTINED {n_unmapped} records with unmapped labels:")
        for lbl, cnt in unmapped_labels.items():
            print(f"    '{lbl}': {cnt}")
        output = output[~unmapped_mask].copy()

    # Count and remove excluded labels (D5-3: Infiltration)
    excluded_mask = output['master_label'] == '__EXCLUDE__'
    n_excluded = excluded_mask.sum()
    if n_excluded > 0:
        excluded_labels = output.loc[excluded_mask, 'original_label'].value_counts()
        print(f"  EXCLUDED {n_excluded} records per spec decision:")
        for lbl, cnt in excluded_labels.items():
            print(f"    '{lbl}': {cnt} (D5-3: statistically unreliable)")
        output = output[~excluded_mask].copy()

    # Summary
    print(f"  Records out: {len(output):,}")
    dist = output['master_label'].value_counts()
    for ml in ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']:
        cnt = dist.get(ml, 0)
        pct = cnt / len(output) * 100 if len(output) > 0 else 0
        print(f"    {ml:>7s}: {cnt:>8,} ({pct:.2f}%)")

    # Flag counts
    n_flagged = output['flag_for_review'].sum()
    n_low_conf = (output['mapping_confidence'] == 'Low').sum()
    print(f"  flag_for_review: {n_flagged} records")
    print(f"  Low confidence : {n_low_conf} records")

    return output


print("\n" + "="*60)
print("  TAXONOMY MAPPING: NSL-KDD")
print("="*60)
nsl_output = apply_taxonomy_and_build_output(
    nsl_X, nsl_original_label, nsl_quality_meta, 'NSL-KDD', map_taxonomy_nsl_kdd
)

print("\n" + "="*60)
print("  TAXONOMY MAPPING: UNSW-NB15")
print("="*60)
unsw_output = apply_taxonomy_and_build_output(
    unsw_X, unsw_original_label, unsw_quality_meta, 'UNSW-NB15', map_taxonomy_unsw_nb15
)

print("\n" + "="*60)
print("  TAXONOMY MAPPING: CICIDS-2017")
print("="*60)
cicids_output = apply_taxonomy_and_build_output(
    cicids_X, cicids_original_label, cicids_quality_meta, 'CICIDS-2017', map_taxonomy_cicids2017
)

# Free memory — feature/label/quality arrays fully consumed into output DataFrames
del nsl_X, nsl_original_label, nsl_quality_meta
del unsw_X, unsw_original_label, unsw_quality_meta
del cicids_X, cicids_original_label, cicids_quality_meta
gc.collect()



  TAXONOMY MAPPING: NSL-KDD
  Records out: 147,907
    NORMALL:   76,967 (52.04%)
       DoSD:   52,985 (35.82%)
      PROBE:   13,954 (9.43%)
    EXPLOIT:    4,001 (2.71%)
    MALWARE:        0 (0.00%)
  flag_for_review: 0 records
  Low confidence : 0 records

  TAXONOMY MAPPING: UNSW-NB15
  EXCLUDED 58871 records per spec decision:
    'Generic': 58871 (D5-3: statistically unreliable)
  Records out: 198,802
    NORMALL:   93,000 (46.78%)
       DoSD:   16,353 (8.23%)
      PROBE:   40,910 (20.58%)
    EXPLOIT:   44,525 (22.40%)
    MALWARE:    4,014 (2.02%)
  flag_for_review: 0 records
  Low confidence : 0 records

  TAXONOMY MAPPING: CICIDS-2017
  EXCLUDED 36 records per spec decision:
    'Infiltration': 36 (D5-3: statistically unreliable)
  Records out: 2,522,326
    NORMALL: 2,096,484 (83.12%)
       DoSD:  321,764 (12.76%)
      PROBE:   90,819 (3.60%)
    EXPLOIT:   11,306 (0.45%)
    MALWARE:    1,953 (0.08%)
  flag_for_review: 0 records
  Low confidence : 0 records


0

---
## Step 8 — Cross-Dataset Master Label Distribution (VC-5.3)
Combined distribution reveals whether MALWARE is statistically viable for SMOTE in Stage 2.

In [31]:
print("\n" + "="*70)
print("  CROSS-DATASET MASTER LABEL DISTRIBUTION")
print("="*70)

all_outputs = {
    'NSL-KDD': nsl_output,
    'UNSW-NB15': unsw_output,
    'CICIDS-2017': cicids_output,
}

# Per-dataset distribution
for ds_name, ds_output in all_outputs.items():
    dist = ds_output['master_label'].value_counts()
    print(f"\n  {ds_name} ({len(ds_output):,} records):")
    for ml in ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']:
        cnt = dist.get(ml, 0)
        print(f"    {ml:>7s}: {cnt:>8,}")

# Combined distribution
combined = pd.concat([nsl_output, unsw_output, cicids_output], ignore_index=True)
combined_dist = combined['master_label'].value_counts()

print(f"\n  COMBINED ({len(combined):,} total records):")
for ml in ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']:
    cnt = combined_dist.get(ml, 0)
    pct = cnt / len(combined) * 100
    print(f"    {ml:>7s}: {cnt:>8,} ({pct:.2f}%)")

# VC-2.4: MALWARE viability check
malware_total = combined_dist.get('MALWARE', 0)
print(f"\n  MALWARE total: {malware_total}")
if malware_total < 500:
    print(f"  WARNING: MALWARE class ({malware_total}) < 500 samples. Review taxonomy.")
else:
    print(f"  MALWARE class sufficient for SMOTE in Stage 2.")

# Source dataset distribution
source_dist = combined['source_dataset'].value_counts()
print(f"\n  Source dataset distribution:")
for ds, cnt in source_dist.items():
    print(f"    {ds}: {cnt:,} ({cnt/len(combined)*100:.1f}%)")

# Flag for review counts
flag_dist = combined.groupby('source_dataset')['flag_for_review'].sum()
print(f"\n  flag_for_review counts:")
for ds, cnt in flag_dist.items():
    print(f"    {ds}: {int(cnt)}")

del combined  # Cell 46 (pipeline summary) rebuilds combined from all_outputs



  CROSS-DATASET MASTER LABEL DISTRIBUTION

  NSL-KDD (147,907 records):
    NORMALL:   76,967
       DoSD:   52,985
      PROBE:   13,954
    EXPLOIT:    4,001
    MALWARE:        0

  UNSW-NB15 (198,802 records):
    NORMALL:   93,000
       DoSD:   16,353
      PROBE:   40,910
    EXPLOIT:   44,525
    MALWARE:    4,014

  CICIDS-2017 (2,522,326 records):
    NORMALL: 2,096,484
       DoSD:  321,764
      PROBE:   90,819
    EXPLOIT:   11,306
    MALWARE:    1,953

  COMBINED (2,869,035 total records):
    NORMALL: 2,266,451 (79.00%)
       DoSD:  391,102 (13.63%)
      PROBE:  145,683 (5.08%)
    EXPLOIT:   59,832 (2.09%)
    MALWARE:    5,967 (0.21%)

  MALWARE total: 5967
  MALWARE class sufficient for SMOTE in Stage 2.

  Source dataset distribution:
    CICIDS-2017: 2,522,326 (87.9%)
    UNSW-NB15: 198,802 (6.9%)
    NSL-KDD: 147,907 (5.2%)

  flag_for_review counts:
    CICIDS-2017: 0
    NSL-KDD: 0
    UNSW-NB15: 0


---
## Step 9 — Save Stage 1 Output as Parquet (Spec Step 7)
Persist the structured output records per dataset. Each Parquet file contains raw features + all 11 mandatory metadata fields.

**D7-2:** Parquet format preserves data types (boolean fields, float precision) and is faster to read.

**D7-3:** Features stored as individual columns (not a single array column).

In [32]:
# Verify output schema compliance before saving
REQUIRED_META_FIELDS = [
    'source_dataset', 'original_label', 'master_label',
    'mapping_confidence', 'flag_for_review',
    'quality_pass', 'quality_flags', 'has_missing', 'bounds_violation',
    'ingestion_timestamp', 'schema_version',
]

for ds_name, ds_output in all_outputs.items():
    missing_fields = [f for f in REQUIRED_META_FIELDS if f not in ds_output.columns]
    if missing_fields:
        raise ValueError(f"{ds_name}: Missing output schema fields: {missing_fields}")
    print(f"{ds_name}: All 11 mandatory output fields present. Columns: {ds_output.shape[1]}")

# Save per-dataset Parquet files
nsl_output.to_parquet('nsl_kdd_stage1_output.parquet', index=False)
print(f"\nSaved nsl_kdd_stage1_output.parquet ({len(nsl_output):,} records, {nsl_output.shape[1]} columns)")

unsw_output.to_parquet('unsw_nb15_stage1_output.parquet', index=False)
print(f"Saved unsw_nb15_stage1_output.parquet ({len(unsw_output):,} records, {unsw_output.shape[1]} columns)")

cicids_output.to_parquet('cicids2017_stage1_output.parquet', index=False)
print(f"Saved cicids2017_stage1_output.parquet ({len(cicids_output):,} records, {cicids_output.shape[1]} columns)")

NSL-KDD: All 11 mandatory output fields present. Columns: 47
UNSW-NB15: All 11 mandatory output fields present. Columns: 53
CICIDS-2017: All 11 mandatory output fields present. Columns: 88

Saved nsl_kdd_stage1_output.parquet (147,907 records, 47 columns)
Saved unsw_nb15_stage1_output.parquet (198,802 records, 53 columns)
Saved cicids2017_stage1_output.parquet (2,522,326 records, 88 columns)


---
## Step 9b — Feature Column Listing Artefact (VC-7.2 / D7-1)
Export per-dataset feature column names so Stage 2 can build the unified feature alignment map
and apply the sentinel -1.0 policy for missing features across datasets.

**CF-NEW-03 / Issue 4:** The actual unified feature space is **154 features** (confirmed at
runtime; spec originally assumed 196). The `feature_alignment_map.json` is the authoritative
source — do not hardcode 154; read it from the artefact.

**D7-1:** Columns absent from a dataset are filled with -1.0 sentinel in Stage 2.
Stage 2 must create a sentinel mask BEFORE normalisation so the model can distinguish
"feature not available" from "feature value is zero".

In [33]:
# VC-7.2: Export per-dataset feature columns for Stage 2 alignment
# Stage 2 uses this to build the unified feature space with -1.0 sentinels
# CF-NEW-03 / Issue 4: confirmed 154 features at runtime (spec assumed 196)
# Authoritative source is feature_alignment_map.json — do not hardcode 154.

feature_columns = {
    'nsl_kdd': sorted([c for c in nsl_output.columns if c not in REQUIRED_META_FIELDS]),
    'unsw_nb15': sorted([c for c in unsw_output.columns if c not in REQUIRED_META_FIELDS]),
    'cicids2017': sorted([c for c in cicids_output.columns if c not in REQUIRED_META_FIELDS]),
}

# Compute the union of all feature columns
all_feature_cols = sorted(
    set(feature_columns['nsl_kdd'])
    | set(feature_columns['unsw_nb15'])
    | set(feature_columns['cicids2017'])
)

feature_alignment_artefact = {
    'schema_version': SCHEMA_VERSION,
    'timestamp': INGESTION_TIMESTAMP,
    'sentinel_value': -1.0,
    'sentinel_policy': (
        'Stage 2 MUST fill missing feature columns with -1.0 sentinel. '
        'A boolean sentinel mask must be created BEFORE normalisation (D7-1) '
        'so the model distinguishes "feature absent" from "feature value is zero".'
    ),
    'total_unique_features': len(all_feature_cols),
    'per_dataset_feature_count': {
        ds: len(cols) for ds, cols in feature_columns.items()
    },
    'per_dataset_features': feature_columns,
    'union_features': all_feature_cols,
}

with open('feature_alignment_map.json', 'w') as f:
    json.dump(feature_alignment_artefact, f, indent=2)

print("Saved feature_alignment_map.json")
print(f"  Total unique features across all datasets: {len(all_feature_cols)}")
for ds, cols in feature_columns.items():
    missing_from_union = len(all_feature_cols) - len(cols)
    print(f"  {ds}: {len(cols)} features ({missing_from_union} will need -1.0 sentinel)")
print(f"  Sentinel value: {feature_alignment_artefact['sentinel_value']}")

Saved feature_alignment_map.json
  Total unique features across all datasets: 154
  nsl_kdd: 36 features (118 will need -1.0 sentinel)
  unsw_nb15: 42 features (112 will need -1.0 sentinel)
  cicids2017: 77 features (77 will need -1.0 sentinel)
  Sentinel value: -1.0


---
## Step 10 — Export Taxonomy Mapping YAML (VC-5.4)
Serialise the taxonomy mapping as a versioned artefact for reproducibility.

In [34]:
# Export taxonomy mapping as a versioned JSON artefact
# (JSON used for Colab compatibility; equivalent to YAML per D5-6)

taxonomy_mapping = {
    "version": SCHEMA_VERSION,
    "timestamp": INGESTION_TIMESTAMP,
    "master_labels": ["NORMALL", "DoSD", "PROBE", "EXPLOIT", "MALWARE"],
    "nsl_kdd": {
        "normal":           {"master_label": "NORMALL", "confidence": "High"},
        "back":             {"master_label": "DoSD",    "confidence": "High"},
        "land":             {"master_label": "DoSD",    "confidence": "High"},
        "neptune":          {"master_label": "DoSD",    "confidence": "High"},
        "pod":              {"master_label": "DoSD",    "confidence": "High"},
        "smurf":            {"master_label": "DoSD",    "confidence": "High"},
        "teardrop":         {"master_label": "DoSD",    "confidence": "High"},
        "apache2":          {"master_label": "DoSD",    "confidence": "High"},
        "udpstorm":         {"master_label": "DoSD",    "confidence": "High"},
        "processtable":     {"master_label": "DoSD",    "confidence": "High"},
        "mailbomb":         {"master_label": "DoSD",    "confidence": "High"},
        "ipsweep":          {"master_label": "PROBE",   "confidence": "High"},
        "nmap":             {"master_label": "PROBE",   "confidence": "High"},
        "portsweep":        {"master_label": "PROBE",   "confidence": "High"},
        "satan":            {"master_label": "PROBE",   "confidence": "High"},
        "mscan":            {"master_label": "PROBE",   "confidence": "High"},
        "saint":            {"master_label": "PROBE",   "confidence": "High"},
        "ftp_write":        {"master_label": "EXPLOIT", "confidence": "High"},
        "guess_passwd":     {"master_label": "EXPLOIT", "confidence": "High"},
        "imap":             {"master_label": "EXPLOIT", "confidence": "High"},
        "multihop":         {"master_label": "EXPLOIT", "confidence": "High"},
        "phf":              {"master_label": "EXPLOIT", "confidence": "High"},
        "spy":              {"master_label": "EXPLOIT", "confidence": "High"},
        "warezclient":      {"master_label": "EXPLOIT", "confidence": "High"},
        "warezmaster":      {"master_label": "EXPLOIT", "confidence": "High"},
        "sendmail":         {"master_label": "EXPLOIT", "confidence": "High"},
        "named":            {"master_label": "EXPLOIT", "confidence": "High"},
        "snmpgetattack":    {"master_label": "EXPLOIT", "confidence": "High"},
        "snmpguess":        {"master_label": "EXPLOIT", "confidence": "High"},
        "xlock":            {"master_label": "EXPLOIT", "confidence": "High"},
        "xsnoop":           {"master_label": "EXPLOIT", "confidence": "High"},
        "worm":             {"master_label": "EXPLOIT", "confidence": "High"},
        "buffer_overflow":  {"master_label": "EXPLOIT", "confidence": "High"},
        "loadmodule":       {"master_label": "EXPLOIT", "confidence": "High"},
        "perl":             {"master_label": "EXPLOIT", "confidence": "High"},
        "rootkit":          {"master_label": "EXPLOIT", "confidence": "High"},
        "httptunnel":       {"master_label": "EXPLOIT", "confidence": "High"},
        "ps":               {"master_label": "EXPLOIT", "confidence": "High"},
        "sqlattack":        {"master_label": "EXPLOIT", "confidence": "High"},
        "xterm":            {"master_label": "EXPLOIT", "confidence": "High"}
    },
    "unsw_nb15": {
        "":                 {"master_label": "NORMALL", "confidence": "High",
                             "note": "Empty string from fillna('') on NaN attack_cat; these are label==0 normal traffic in the Kaggle pre-split version"},
        "Normal":           {"master_label": "NORMALL", "confidence": "High"},
        "DoS":              {"master_label": "DoSD",    "confidence": "High"},
        "Reconnaissance":   {"master_label": "PROBE",   "confidence": "High"},
        "Fuzzers":          {"master_label": "PROBE",   "confidence": "Medium", "decision": "D5-1"},
        "Analysis":         {"master_label": "PROBE",   "confidence": "Medium", "decision": "D5-2"},
        "Exploits":         {"master_label": "EXPLOIT", "confidence": "High"},
        "Generic":          {"action": "EXCLUDE", "decision": "D5-6",
                             "reason": "~58,871 samples; no agreed real-world threat analogue; excluded per D5-6"},
        "Backdoors":        {"master_label": "MALWARE", "confidence": "High"},
        "Shellcode":        {"master_label": "MALWARE", "confidence": "High"},
        "Worms":            {"master_label": "MALWARE", "confidence": "High"}
    },
    "cicids2017": {
        "BENIGN":           {"master_label": "NORMALL", "confidence": "High"},
        "DoS slowloris":    {"master_label": "DoSD",    "confidence": "High"},
        "DoS Slowhttptest": {"master_label": "DoSD",    "confidence": "High"},
        "DoS Hulk":         {"master_label": "DoSD",    "confidence": "High"},
        "DoS GoldenEye":    {"master_label": "DoSD",    "confidence": "High"},
        "DDoS":             {"master_label": "DoSD",    "confidence": "High"},
        "PortScan":         {"master_label": "PROBE",   "confidence": "High"},
        "FTP-Patator":      {"master_label": "EXPLOIT", "confidence": "High"},
        "SSH-Patator":      {"master_label": "EXPLOIT", "confidence": "High"},
        "Web Attack - Brute Force": {"master_label": "EXPLOIT", "confidence": "High"},
        "Web Attack - XSS":         {"master_label": "EXPLOIT", "confidence": "High"},
        "Web Attack - Sql Injection":{"master_label": "EXPLOIT", "confidence": "High"},
        "Heartbleed":       {"master_label": "EXPLOIT", "confidence": "High", "decision": "D5-5"},
        "Infiltration":     {"action": "EXCLUDE", "decision": "D5-3", "reason": "~36 samples; statistically unreliable"},
        "Bot":              {"master_label": "MALWARE", "confidence": "High"}
    },
    "ambiguity_decisions": {
        "D5-1": "Fuzzers (UNSW-NB15) -> PROBE. Fuzzing is primarily reconnaissance behaviour.",
        "D5-2": "Analysis (UNSW-NB15) -> PROBE. Includes port/service enumeration.",
        "D5-3": "Infiltration (CICIDS-2017) -> EXCLUDED. ~36 samples, statistically unreliable.",
        "D5-4": "Generic (UNSW-NB15) — SUPERSEDED by D5-6. Provisional EXPLOIT/Low mapping removed.",
        "D5-5": "Heartbleed (CICIDS-2017) -> EXPLOIT. Memory disclosure CVE, not malware.",
        "D5-6": "Generic (UNSW-NB15) -> EXCLUDED. ~58,871 records (22.8% of UNSW-NB15). No agreed real-world threat analogue — the UNSW-NB15 'Generic' category uses crafted packets that do not correspond to a distinct attack class in any recognised taxonomy. Excluding avoids injecting 22.8% low-quality signal into the EXPLOIT class.",
    },
    "data_handling_notes": {
        "unsw_empty_attack_cat": (
            "UNSW-NB15 Kaggle pre-split: rows with NaN attack_cat have label==0 (normal). "
            "fillna('') converts these to empty string. The '' key maps to NORMALL. "
            "This is intentional and semantically correct."
        ),
        "infiltration_excluded": (
            "CICIDS-2017 Infiltration excluded per D5-3 (~36 samples). "
            "Records are removed after taxonomy mapping, not quarantined."
        ),
        "generic_excluded": (
            "UNSW-NB15 Generic excluded per D5-6 (~58,871 samples, 22.8% of UNSW-NB15). "
            "Records are removed after taxonomy mapping. "
            "UNSW-NB15 output: ~257,673 -> ~198,802 records. "
            "Combined corpus: ~2,927,906 -> ~2,869,035 records."
        ),
        "MF-NEW-04_cicids_duplicates": (
            "CICIDS-2017 has ~9.66% duplicate records — typical CICFlowMeter regeneration artefact. "
            "These are removed by QG-01. Effective yield is ~90.3% of raw data. Not a data error."
        ),
        "MF-NEW-01_cicids_bounds_violation": (
            "~98-100% of CICIDS-2017 records flagged bounds_violation=True due to CICFlowMeter "
            "integer overflow (e.g. Fwd Header Length, Init_Win_bytes_forward > 2^31). "
            "Stage 2 MUST clip these rather than exclude records."
        )
    }
}

with open('taxonomy_mapping_v1.0.json', 'w') as f:
    json.dump(taxonomy_mapping, f, indent=2)

print("Saved taxonomy_mapping_v1.0.json")
print(f"  Version: {taxonomy_mapping['version']}")
print(f"  NSL-KDD mappings: {len(taxonomy_mapping['nsl_kdd'])}")
print(f"  UNSW-NB15 mappings: {len(taxonomy_mapping['unsw_nb15'])} (includes '' for empty attack_cat, Generic excluded)")
print(f"  CICIDS-2017 mappings: {len(taxonomy_mapping['cicids2017'])}")
print(f"  Ambiguity decisions: {len(taxonomy_mapping['ambiguity_decisions'])} (D5-1 through D5-6)")
print(f"  Data handling notes: {len(taxonomy_mapping['data_handling_notes'])}")


Saved taxonomy_mapping_v1.0.json
  Version: 1.0
  NSL-KDD mappings: 40
  UNSW-NB15 mappings: 11 (includes '' for empty attack_cat, Generic excluded)
  CICIDS-2017 mappings: 15
  Ambiguity decisions: 6 (D5-1 through D5-6)
  Data handling notes: 5


---
## Step 11 — Save QG-03 Artefact
Serialise the Infinity replacement values so Stage 2 can verify and recompute.

**Note (MF-07 / VC-4.4):** These replacement values were computed on the full dataset in this notebook. Stage 2 MUST recompute on train-only data after splitting to avoid data leakage.

**Structural fix:** `apply_quality_gate()` now accepts an `inf_replacement_values` parameter. Stage 2 should compute train-only p99 values and pass them — this eliminates the leakage vector entirely.

In [35]:
# Save QG-03 Infinity replacement artefacts
# Issue 1 (QG-03 leakage): These p99 values are computed on the FULL dataset because
# Stage 1 does not own the train/test split. This is a KNOWN data leakage vector.
# Stage 2 MUST recompute replacement values using train-only data after splitting,
# then pass them via apply_quality_gate(..., inf_replacement_values={col: val}).
qg03_artefact = {
    'schema_version': SCHEMA_VERSION,
    'timestamp': INGESTION_TIMESTAMP,
    'WARNING': (
        'LEAKAGE VECTOR — These replacement values were computed on the FULL dataset '
        '(train+test combined). Stage 2 MUST recompute percentiles on train-only split '
        'before applying to test data. See VC-4.4 / MF-07.'
    ),
    'computation_scope': 'full_dataset',
    'leakage_acknowledged': True,
    'recompute_required_by': 'Stage 2',
    'structural_fix_available': (
        'apply_quality_gate() accepts inf_replacement_values parameter. '
        'Stage 2 should compute train-only p99 values and pass them to '
        'eliminate leakage entirely.'
    ),
    'nsl_kdd': nsl_qg_report.get('inf_replacements', {}),
    'unsw_nb15': unsw_qg_report.get('inf_replacements', {}),
    'cicids2017': cicids_qg_report.get('inf_replacements', {}),
}

with open('qg03_replacement_values.json', 'w') as f:
    json.dump(qg03_artefact, f, indent=2)

print("Saved qg03_replacement_values.json")
print("  ⚠ WARNING: Values computed on full dataset — Stage 2 must recompute on train-only split.")
print("  ✓ Structural fix available: pass inf_replacement_values to apply_quality_gate()")
for ds in ['nsl_kdd', 'unsw_nb15', 'cicids2017']:
    n = len(qg03_artefact[ds])
    print(f"  {ds}: {n} columns with Infinity replacements")

Saved qg03_replacement_values.json
  ⚠ WARNING: Values computed on full dataset — Stage 2 must recompute on train-only split.
  ✓ Structural fix available: pass inf_replacement_values to apply_quality_gate()
  nsl_kdd: 0 columns with Infinity replacements
  unsw_nb15: 0 columns with Infinity replacements
  cicids2017: 2 columns with Infinity replacements


---
## Step 12 — Pipeline Summary Report (VC-6.3)
Structured summary of the entire Stage 1 pipeline run.

In [36]:
print("\n" + "#"*70)
print("#" + " "*18 + "STAGE 1 PIPELINE SUMMARY" + " "*18 + "  #")
print("#"*70)

print(f"\nSchema Version    : {SCHEMA_VERSION}")
print(f"Ingestion Time    : {INGESTION_TIMESTAMP}")
print(f"Random State      : {RANDOM_STATE}")

# Issue 4 fix: Log which CICIDS-2017 files were actually loaded
print(f"\n--- CICIDS-2017 Files Loaded ({len(cicids_csv_files)}) ---")
for f in cicids_csv_files:
    print(f"    {f}")

print(f"\n--- Per-Dataset Record Counts ---")
qg_reports = {
    'NSL-KDD': nsl_qg_report,
    'UNSW-NB15': unsw_qg_report,
    'CICIDS-2017': cicids_qg_report,
}

for ds_name, rpt in qg_reports.items():
    output_len = len(all_outputs[ds_name])
    dup_pct = rpt['duplicates_removed'] / rpt['records_in'] * 100 if rpt['records_in'] > 0 else 0
    print(f"\n  {ds_name}:")
    print(f"    Raw records loaded   : {rpt['records_in']:,}")
    print(f"    Duplicates removed   : {rpt['duplicates_removed']:,} ({dup_pct:.2f}%)")
    print(f"    Quarantined (QG)     : {rpt['quarantined']:,} ({rpt['quarantine_rate']*100:.2f}%)")
    print(f"    Stage 1 output       : {output_len:,}")

# MF-NEW-01 / MF-NEW-04: Per-dataset QG flag rates for traceability
print(f"\n--- Per-Dataset QG Flag Rates (bounds_violation) ---")
for ds_name in ['NSL-KDD', 'UNSW-NB15', 'CICIDS-2017']:
    ds_data = all_outputs[ds_name]
    if 'bounds_violation' in ds_data.columns:
        bv_count = ds_data['bounds_violation'].sum()
        bv_pct = bv_count / len(ds_data) * 100 if len(ds_data) > 0 else 0
        print(f"  {ds_name}: {int(bv_count):,} / {len(ds_data):,} ({bv_pct:.1f}%)")
        if bv_pct > 90:
            print(f"    ⚠ CICFlowMeter integer overflow artefact — Stage 2 must clip, not exclude")

print(f"\n--- Combined Master Label Distribution ---")
combined = pd.concat(list(all_outputs.values()), ignore_index=True)
combined_dist = combined['master_label'].value_counts()
print(f"  Total records: {len(combined):,}")
for ml in ['NORMALL', 'DoSD', 'PROBE', 'EXPLOIT', 'MALWARE']:
    cnt = combined_dist.get(ml, 0)
    pct = cnt / len(combined) * 100
    print(f"    {ml:>7s}: {cnt:>8,} ({pct:.2f}%)")

print(f"\n--- Source Dataset Distribution ---")
for ds, cnt in combined['source_dataset'].value_counts().items():
    print(f"    {ds}: {cnt:,} ({cnt/len(combined)*100:.1f}%)")

print(f"\n--- flag_for_review Summary ---")
total_flagged = combined['flag_for_review'].sum()
print(f"  Total flagged: {int(total_flagged)} ({total_flagged/len(combined)*100:.2f}%)")
for ds in ['NSL-KDD', 'UNSW-NB15', 'CICIDS-2017']:
    ds_data = combined[combined['source_dataset'] == ds]
    ds_flagged = ds_data['flag_for_review'].sum()
    print(f"    {ds}: {int(ds_flagged)}")

print(f"\n--- Mapping Confidence Distribution ---")
conf_dist = combined['mapping_confidence'].value_counts()
for conf in ['High', 'Medium', 'Low']:
    cnt = conf_dist.get(conf, 0)
    print(f"    {conf:>6s}: {cnt:>8,}")

# D5-6: Log Generic exclusion count for traceability
unsw_qg_in = unsw_qg_report.get('records_in', 0)
unsw_out = len(all_outputs['UNSW-NB15'])
generic_excluded = unsw_qg_in - unsw_qg_report.get('duplicates_removed', 0) - unsw_qg_report.get('quarantined', 0) - unsw_out
if generic_excluded > 0:
    print(f"\n--- D5-6: Generic Records Excluded (UNSW-NB15) ---")
    print(f"  ~{generic_excluded:,} Generic records removed (no agreed real-world threat analogue)")
    print(f"  UNSW-NB15 output after exclusion: {unsw_out:,} records")

print(f"\n--- Output Artefacts ---")
print(f"  nsl_kdd_stage1_output.parquet")
print(f"  unsw_nb15_stage1_output.parquet")
print(f"  cicids2017_stage1_output.parquet")
print(f"  taxonomy_mapping_v1.0.json")
print(f"  qg03_replacement_values.json")
print(f"  feature_alignment_map.json")
if len(nsl_quarantine) > 0:
    print(f"  nsl_kdd_quarantine.parquet")
if len(unsw_quarantine) > 0:
    print(f"  unsw_nb15_quarantine.parquet")
if len(cicids_quarantine) > 0:
    print(f"  cicids2017_quarantine.parquet")

print(f"\n--- Stage Boundary Reminder ---")
print(f"  Stage 1 output contains RAW features — no encoding, scaling, or imputation.")
print(f"  Stage 2 is responsible for: OHE/encoding, imputation, scaling, train/test split,")



######################################################################
#                  STAGE 1 PIPELINE SUMMARY                    #
######################################################################

Schema Version    : 1.0
Ingestion Time    : 2026-03-11T02:32:41.807928+00:00
Random State      : 42

--- CICIDS-2017 Files Loaded (8) ---
    cicids2017_dataset\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
    cicids2017_dataset\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
    cicids2017_dataset\Friday-WorkingHours-Morning.pcap_ISCX.csv
    cicids2017_dataset\Monday-WorkingHours.pcap_ISCX.csv
    cicids2017_dataset\Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
    cicids2017_dataset\Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
    cicids2017_dataset\Tuesday-WorkingHours.pcap_ISCX.csv
    cicids2017_dataset\Wednesday-workingHours.pcap_ISCX.csv

--- Per-Dataset Record Counts ---

  NSL-KDD:
    Raw records loaded   : 148,517
    Duplicates removed

---
## Stage 2 Forward Requirements

The following requirements are raised by Stage 1 for Stage 2 implementation:

1. **Sentinel mask for -1.0:** Must implement sentinel mask before normalisation (D7-1). Use `feature_alignment_map.json` for the union feature list.
2. **Schema version validation:** Must validate `schema_version` on first record read (Step 7 risk)
3. **Dual stratification:** Must use `master_label x source_dataset` for train/test split (Step 5)
4. **QG-03 recomputation:** Must recompute 99th percentile Infinity replacements on **train-only** data (MF-07). The `apply_quality_gate()` function accepts an `inf_replacement_values` parameter — pass the train-only computed dict to eliminate leakage.
5. **has_missing handling:** Must not apply SMOTE to records where `has_missing = True` without imputation
6. **bounds_violation handling:** Must clip or correct before scaling. **MF-NEW-01:** For CICIDS-2017, ~98-100% of records are flagged `bounds_violation=True` due to CICFlowMeter integer overflow artefacts (e.g., `Fwd Header Length`, `Init_Win_bytes_forward` containing values > 2³¹). Stage 2 MUST clip these columns to domain-valid ranges rather than excluding records. Consider using `np.iinfo(np.int32).max` as an upper clip boundary for known CICFlowMeter overflow columns.
7. **flag_for_review records:** Must explicitly log count in train and test splits
8. **Categorical encoding:** OHE, target encoding, or frequency encoding is Stage 2's decision
9. **Feature alignment:** Load `feature_alignment_map.json` for the authoritative union of all feature columns. **Issue 4:** Runtime-confirmed count is **154 features** (spec assumed 196). Do not hardcode — read from artefact. Apply -1.0 sentinel policy for missing columns per dataset.
10. **D5-6 — Generic exclusion (UNSW-NB15):** ~58,871 Generic records were **excluded** in Stage 1 (no agreed real-world threat analogue). UNSW-NB15 output is ~198,802 records; combined corpus is ~2,869,035. All remaining records have `mapping_confidence` of High or Medium only — no Low-confidence records remain. No further action required in Stage 2.
11. **MF-NEW-04 — CICIDS-2017 duplicate rate:** ~9.66% of CICIDS-2017 records are duplicates (CICFlowMeter regeneration artefact). These are removed by QG-01. Effective yield is ~90.3% of raw CICIDS-2017 records. This is expected CICFlowMeter behaviour, not a data error.
12. **Issue 3 — CICIDS-2017 near-zero variance features:** 12 features with unique ≤ 2 or var < 0.01 are preserved in Stage 1 (raw features MUST not be dropped here). Stage 2 MUST apply variance filtering before scaling to remove these: `Bwd PSH Flags`, `Fwd URG Flags`, `Bwd URG Flags`, `RST Flag Count`, `CWE Flag Count`, `ECE Flag Count`, `Fwd Avg Bytes/Bulk`, `Fwd Avg Packets/Bulk`, `Fwd Avg Bulk Rate`, `Bwd Avg Bytes/Bulk`, `Bwd Avg Packets/Bulk`, `Bwd Avg Bulk Rate`. These columns contribute no discriminative signal and inflate dimensionality.

**Stage 3 (VAE) handles dimensionality reduction to QUANTUM_DIM features — PCA must NOT be used.**

---
## Version Control Checklist (VC-1.1)

After successful execution, commit all artefacts to version control:

| Artefact | Purpose | Status |
|----------|---------|--------|
| `Stage1_Preprocessing_Complete.ipynb` | Pipeline notebook (with execution outputs) | Pending execution |
| `nsl_kdd_stage1_output.parquet` | NSL-KDD structured output | Produced at runtime |
| `unsw_nb15_stage1_output.parquet` | UNSW-NB15 structured output | Produced at runtime |
| `cicids2017_stage1_output.parquet` | CICIDS-2017 structured output | Produced at runtime |
| `taxonomy_mapping_v1.0.json` | Versioned taxonomy mapping (VC-5.4) | Produced at runtime |
| `qg03_replacement_values.json` | QG-03 infinity replacements (leakage-flagged) | Produced at runtime |
| `feature_alignment_map.json` | Feature column listing for sentinel policy (VC-7.2) | Produced at runtime |
| `*_quarantine.parquet` | Quarantine records (if any) | Produced at runtime |

**Tag:** `stage1-v1.1` after successful execution and review.
